# Large Files: Stream Rows and Chunks

A CSV file larger than memory cannot be read with `CSV.File`. CSV.jl offers two ways to process it piece by piece:

- `CSV.Rows` iterates one row at a time in constant memory.
- `CSV.Chunks` parses the file in typed blocks, one at a time.

This tutorial generates a 100,000-row file, which easily fits in memory, so the results can be checked against a full read. It needs CSV.jl and Tables.jl.

In [ ]:
using CSV, Tables, Statistics, Random

Random.seed!(1)
cd(mktempdir())
open("trips.csv", "w") do io
	println(io, "id,city,distance,fare")
	for i in 1:100_000
		d = round(rand() * 20, digits = 2)
		println(io, i, ",", rand(["Lisbon", "Porto", "Faro"]), ",", d, ",", rand() < 0.01 ? "" : string(round(3 + 0.9 * d, digits = 2)))
	end
end
println(round(filesize("trips.csv") / 2^20, digits = 1), " MB")

## Stream Rows

With `reusebuffer = true`, `CSV.Rows` keeps one row in memory at a time. Pass `types` (one per column) so the values arrive already parsed: calling `parse` on each string yourself is many times slower. Wrapping the loop in a function keeps the running totals local and fast.

In [ ]:
function fare_by_city(path)
	totals = Dict{String, Float64}()
	counts = Dict{String, Int}()
	for row in CSV.Rows(path; types = [Int, String, Float64, Float64], reusebuffer = true)
		ismissing(row.fare) && continue
		totals[row.city] = get(totals, row.city, 0.0) + row.fare
		counts[row.city] = get(counts, row.city, 0) + 1
	end
	Dict(c => totals[c] / counts[c] for c in keys(totals))
end

fare_by_city("trips.csv")  # first call compiles
@time fare_by_city("trips.csv")

## Untyped Rows

Without `types`, every value is a string view into the file buffer (or `missing`).

In [ ]:
row = first(CSV.Rows("trips.csv"))
(row.id, row.city, row.distance, row.fare)

## Filter to a Smaller File

`CSV.write` accepts any iterator of rows, so a filtered stream goes straight to a new file without loading the original.

In [ ]:
CSV.write("long_trips.csv", Iterators.filter(r -> r.distance > 19.5, CSV.Rows("trips.csv"; types = [Int, String, Float64, Float64])))
println(length(CSV.File("long_trips.csv")), " long trips")

## Chunks

`CSV.Chunks` splits the file into `ntasks` byte ranges. Each chunk is a typed `CSV.File`. Compute a small summary per chunk and combine them at the end.

In [ ]:
summaries = map(CSV.Chunks("trips.csv"; ntasks = 8)) do chunk
	fare = collect(skipmissing(Tables.getcolumn(chunk, :fare)))
	(rows = length(chunk), n = length(fare), total = sum(fare), longest = maximum(Tables.getcolumn(chunk, :distance)))
end
foreach(println, summaries)

In [ ]:
println("Rows: ", sum(s.rows for s in summaries))
println("Mean fare: ", sum(s.total for s in summaries) / sum(s.n for s in summaries))
println("Longest trip: ", maximum(s.longest for s in summaries))

## Check Against a Full Read

The file fits in memory here, so the streamed results can be verified.

In [ ]:
full = Tables.columntable(CSV.File("trips.csv"))
println("Rows: ", length(full.id))
println("Mean fare: ", mean(skipmissing(full.fare)))
println("Longest trip: ", maximum(full.distance))

## Timing

Each call is timed after a first, compiling run. Parallel parsing in `CSV.File` needs Julia started with several threads (`julia --threads=auto`).

In [ ]:
readers = [
	"CSV.File"   => () -> CSV.File("trips.csv"),
	"CSV.Rows"   => () -> count(_ -> true, CSV.Rows("trips.csv"; reusebuffer = true)),
	"CSV.Chunks" => () -> sum(length, CSV.Chunks("trips.csv"; ntasks = 8)),
]
for (name, f) in readers
	f()
	println(rpad(name, 12), round(@elapsed(f()), digits = 3), " s")
end
println("Threads: ", Threads.nthreads())

## Next Steps

- The panel's **Stream Rows** and **Chunks** models generate this code for your own file.
- For repeated queries on large data, load the CSV into DuckDB and use SQL (see the SQL Databases panel).